In [4]:
from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import CharacterTextSplitter
from langchain_community.vectorstores import FAISS
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_groq import ChatGroq

from langgraph.graph import StateGraph
from typing import TypedDict
import os 
from dotenv import load_dotenv
load_dotenv()

# Load txt
loader = TextLoader("../documents/data.txt")
docs = loader.load()

# Split text
splitter = CharacterTextSplitter(chunk_size=200, chunk_overlap=20)
documents = splitter.split_documents(docs)

# Embeddings
embeddings = HuggingFaceEmbeddings()

# Vector DB
vectorstore = FAISS.from_documents(documents, embeddings)

retriever = vectorstore.as_retriever()

# LLM
llm = ChatGroq(
    model="llama-3.1-70b-versatile",
    GROQ_API_KEY=os.getenv("GROQ_API_KEY")
)

# State
class State(TypedDict):
    question: str
    context: str
    answer: str


# Retriever Agent
def retriever_agent(state):

    docs = retriever.invoke(state["question"])

if docs:

    context = docs[0].page_content
else:
    context = "No relevant information found."




# Answer Agent
from langchain_core.messages import HumanMessage
def answer_agent(state):

    prompt = f"""
    Answer the question using this context.

    Context:
    {state['context']}

    Question:
    {state['question']}
    """

    response = llm.invoke([HumanMessage(content=prompt)])

    return {"answer": response.content}


# Graph
builder = StateGraph(State)

builder.add_node("retriever", retriever_agent)
builder.add_node("answer", answer_agent)

builder.set_entry_point("retriever")

builder.add_edge("retriever", "answer")

builder.set_finish_point(" answer")

graph = builder.compile()

# Run
question = input("Ask: ")


result = graph.invoke({"question": question})

print("\nAI",result["answer"])

RuntimeError: Error loading ../documents/data.txt